In [1]:
# Cell 1 — Setup
import os
import sys

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

# Add repo root to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from dotenv import load_dotenv
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(repo_root, ".env"))

print("✅ Setup complete")
print(f"Repo root: {repo_root}")

✅ Setup complete
Repo root: /Users/abhinav/Documents/GitHub/IITB_Capstone


In [2]:
# Cell 2 — Load pipeline and RAG modules
from modules.pipeline import generate_posts, UNIFIED_TEMPLATE, get_examples
from modules.rag import build_index, retrieve_brand_context, is_index_built, clear_index

print("✅ Pipeline loaded")
print("✅ RAG module loaded")

✅ Pipeline loaded
✅ RAG module loaded


In [3]:
# Cell 3 — Test configuration
BRAND    = "Adobe"
TOPIC    = "Adobe's acquisition of Semrush"
TONE     = "professional, strategic, forward looking"
PLATFORM = "linkedin"
PDF_PATH = "../../data/brand_guidelines/Adobe_Brand_Voice.pdf"

print(f"Brand:    {BRAND}")
print(f"Topic:    {TOPIC}")
print(f"Tone:     {TONE}")
print(f"Platform: {PLATFORM}")
print(f"PDF:      {PDF_PATH}")

Brand:    Adobe
Topic:    Adobe's acquisition of Semrush
Tone:     professional, strategic, forward looking
Platform: linkedin
PDF:      ../../data/brand_guidelines/Adobe_Brand_Voice.pdf


In [4]:
# Cell 4 — Generate WITHOUT brand context
print("=" * 60)
print("WITHOUT RAG — No brand context")
print("=" * 60)
print()

variants_no_rag = generate_posts(
    brand_name=BRAND,
    topic=TOPIC,
    tone=TONE,
    platform=PLATFORM,
    pdf_path=None
)

if variants_no_rag:
    no_rag_post = variants_no_rag[0]
    print("VARIANT 1 (first generated):")
    print()
    print(no_rag_post["post_text"])
    print()
    print(f"Hashtags: {' '.join(no_rag_post['hashtags'])}")
    print()
    print(f"Reasoning: {no_rag_post.get('reasoning', 'N/A')[:150]}...")
else:
    print("❌ Generation failed")

WITHOUT RAG — No brand context

VARIANT 1 (first generated):

Today, Adobe completed its acquisition of Semrush—a transformative move that strengthens our position in the AI-first economy.

This strategic expansion gives businesses unified access to creative, marketing, and commerce solutions. As AI-driven discovery channels evolve, brands need integrated tools to appear, engage, and convert across every surface.

We're not just acquiring technology. We're building the platform that helps businesses thrive when the rules of customer discovery are fundamentally changing. Together with Semrush, we're accelerating growth for millions of marketers and creators worldwide.

Hashtags: #AdobeAcquisition #DigitalTransformation #MarketingInnovation #ArtificialIntelligence

Reasoning: Audience: C-suite and business leaders. Key message: Strategic expansion driving competitive advantage. Leads with business impact and market position...


In [5]:
# Cell 5 — Build RAG index and show retrieved context
print("Building RAG index for Adobe...")

# Clear any existing index for clean test
clear_index(BRAND)

# Build fresh index
build_index(PDF_PATH, BRAND)

# Retrieve context — same query Claude will use
context = retrieve_brand_context(
    brand_name=BRAND,
    topic=TOPIC,
    platform=PLATFORM,
    tone=TONE
)

print()
print("=" * 60)
print("RETRIEVED BRAND CONTEXT (injected into prompt)")
print("=" * 60)
print(context)
print()
print(f"Total characters retrieved: {len(context)}")

Building RAG index for Adobe...
✅ Index cleared for Adobe
Building RAG index for Adobe...
  Extracted 5493 characters from PDF
  Created 13 chunks
Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded!
  Generated embeddings shape: (13, 384)
✅ RAG index built for Adobe (13 chunks indexed)

RETRIEVED BRAND CONTEXT (injected into prompt)
ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity

, optimistic. We encourage action and believe the world is full of possibilities. Progressive We are forward-thinking and speak with conviction. We feel cutting edge and culturally relevant. We set trends — we do not chase them. Creative We are original, fresh, and modern. Open-minded and curious. Clever and unexpected. We demonstrate 

In [6]:
# Cell 6 — Generate WITH brand context
print("=" * 60)
print("WITH RAG — Brand context from Adobe_Brand_Voice.pdf")
print("=" * 60)
print()

variants_with_rag = generate_posts(
    brand_name=BRAND,
    topic=TOPIC,
    tone=TONE,
    platform=PLATFORM,
    pdf_path=PDF_PATH
)

if variants_with_rag:
    rag_post = variants_with_rag[0]
    print("VARIANT 1 (first generated):")
    print()
    print(rag_post["post_text"])
    print()
    print(f"Hashtags: {' '.join(rag_post['hashtags'])}")
    print()
    print(f"Reasoning: {rag_post.get('reasoning', 'N/A')[:150]}...")
else:
    print("❌ Generation failed")

WITH RAG — Brand context from Adobe_Brand_Voice.pdf

✅ Using existing RAG index for Adobe
✅ Retrieved brand context (1503 chars)
VARIANT 1 (first generated):

Today, Adobe completed its acquisition of Semrush, marking a significant step in how we help businesses thrive in an AI-first world.

This move expands our ability to help companies show up where their customers are looking—across search, social, and emerging AI surfaces. With AI-driven traffic to U.S. retail sites up 269% year-over-year, the competitive advantage goes to those who can be found, understood, and engaged at scale.

Semrush's industry-leading insights complement Adobe's creative and commerce capabilities, enabling marketers to not just create better experiences, but ensure those experiences reach the right people at the right moment.

This is about empowering every business to stand out and succeed in a rapidly evolving digital landscape.

Hashtags: #AdobeAcquisition #DigitalMarketing #AIFirst #MarketingTech

Reason

In [7]:
# Cell 7 — Side by side comparison
print("SIDE BY SIDE COMPARISON")
print()
print("WITHOUT RAG:")
print("-" * 40)
if variants_no_rag:
    print(variants_no_rag[0]["post_text"][:300])
print()
print("WITH RAG:")
print("-" * 40)
if variants_with_rag:
    print(variants_with_rag[0]["post_text"][:300])

SIDE BY SIDE COMPARISON

WITHOUT RAG:
----------------------------------------
Today, Adobe completed its acquisition of Semrush—a transformative move that strengthens our position in the AI-first economy.

This strategic expansion gives businesses unified access to creative, marketing, and commerce solutions. As AI-driven discovery channels evolve, brands need integrated tool

WITH RAG:
----------------------------------------
Today, Adobe completed its acquisition of Semrush, marking a significant step in how we help businesses thrive in an AI-first world.

This move expands our ability to help companies show up where their customers are looking—across search, social, and emerging AI surfaces. With AI-driven traffic to U


In [8]:
# Cell 8 — Scoring rubric
criteria = [
    "Uses Adobe-specific vocabulary and values",
    "Tone matches Adobe brand voice",
    "Mentions Adobe themes (creativity, AI, enterprise)",
    "Sounds like Adobe — not a generic tech company",
    "Post would feel at home on Adobe's actual LinkedIn page",
]

print("BRAND VOICE ALIGNMENT SCORING (fill in 1-5 per criterion)")
print()
print(f"{'Criteria':<50} {'No RAG':>8} {'With RAG':>10}")
print("-" * 70)
for c in criteria:
    print(f"{c:<50} {'?':>8} {'?':>10}")

print()
print("Expected result:")
print("With RAG scores consistently higher — especially on")
print("'Uses Adobe-specific vocabulary' and 'Sounds like Adobe'.")
print()
print("Exam answer:")
print("RAG retrieves Adobe-specific language from their brand guidelines")
print("and injects it into Claude's prompt. Without RAG, posts are")
print("professional but generic. With RAG, posts use Adobe's actual")
print("vocabulary and themes — demonstrating measurable improvement")
print("in brand voice alignment.")

BRAND VOICE ALIGNMENT SCORING (fill in 1-5 per criterion)

Criteria                                             No RAG   With RAG
----------------------------------------------------------------------
Uses Adobe-specific vocabulary and values                 ?          ?
Tone matches Adobe brand voice                            ?          ?
Mentions Adobe themes (creativity, AI, enterprise)        ?          ?
Sounds like Adobe — not a generic tech company            ?          ?
Post would feel at home on Adobe's actual LinkedIn page        ?          ?

Expected result:
With RAG scores consistently higher — especially on
'Uses Adobe-specific vocabulary' and 'Sounds like Adobe'.

Exam answer:
RAG retrieves Adobe-specific language from their brand guidelines
and injects it into Claude's prompt. Without RAG, posts are
professional but generic. With RAG, posts use Adobe's actual
vocabulary and themes — demonstrating measurable improvement
in brand voice alignment.


In [9]:
# Cell 9 — Test with Adidas
ADIDAS_PDF = "../../data/brand_guidelines/Adidas_Brand_Voice.pdf"

print("=" * 60)
print("ADIDAS — INSTAGRAM — WITH vs WITHOUT RAG")
print("=" * 60)

# Without RAG
adidas_no_rag = generate_posts(
    brand_name="Adidas",
    topic="launching new running shoe",
    tone="bold, performance-driven, athlete-focused",
    platform="instagram",
    pdf_path=None
)

# With RAG
clear_index("Adidas")
adidas_with_rag = generate_posts(
    brand_name="Adidas",
    topic="launching new running shoe",
    tone="bold, performance-driven, athlete-focused",
    platform="instagram",
    pdf_path=ADIDAS_PDF
)

print("WITHOUT RAG:")
if adidas_no_rag:
    print(adidas_no_rag[0]["post_text"][:250])
print()
print("WITH RAG:")
if adidas_with_rag:
    print(adidas_with_rag[0]["post_text"][:250])

ADIDAS — INSTAGRAM — WITH vs WITHOUT RAG
✅ Index cleared for Adidas
Building RAG index for Adidas...
  Extracted 6170 characters from PDF
  Created 14 chunks
  Generated embeddings shape: (14, 384)
✅ RAG index built for Adidas (14 chunks indexed)
✅ Retrieved brand context (1504 chars)
WITHOUT RAG:
We just redefined what fast looks like. 🚀 The new Ultraboost Pro isn't just another shoe—it's a paradigm shift in running technology. With our most responsive cushioning system yet, athletes are already clocking faster splits. This is what happens wh

WITH RAG:
We broke our own records building this. 🏃‍♂️

Faster toe-off. Better energy return. 3% speed improvement in field testing.

The new Ultraboost Pro isn't hype. It's engineered for runners who refuse to settle. Your personal best is just a starting poi
